In [ ]:
# 1) Instalando o que preciso pra rodar Spark + Delta Lake
!pip install pyspark delta-spark -q

# 2) Importando as libs que vou usar ao longo do notebook
import pyspark
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, abs, when, round

# 3) Criando a Spark Session com suporte ao Delta
# A ideia aqui é simular localmente um ambiente parecido com o Databricks
builder = pyspark.sql.SparkSession.builder.appName("Analytics_Reconciliation_Engine") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Deixo só os erros aparecendo no log, senão o Spark enche a tela de INFO
spark.sparkContext.setLogLevel("ERROR")

print("Ambiente configurado. Spark Session rodando com suporte ao Delta Lake.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.3 MB/s eta 0:00:00
Ambiente configurado. Spark Session rodando com suporte ao Delta Lake.


## 1. Simulação do Cenário de Migração e Geração de Dados

Nesta etapa, estamos simulando o momento crítico de uma migração de *Data Warehouse*. Temos duas fontes de dados:
1. **Sistema Legado (Fonte da Verdade):** A base antiga que atualmente alimenta os painéis do Power BI.
2. **Databricks Gold Layer (Novo Ambiente):** A nova base modelada no Delta Lake (Star Schema) que precisa ser validada antes de receber o apontamento do BI.

Para testar nosso **Motor de Reconciliação**, geramos dados sintéticos agregados (Data Marts) e injetamos duas anomalias intencionais na base Gold:
* Uma divergência de valor financeiro (erro de arredondamento/conversão).
* Uma quebra de volumetria (uma linha de agregação ausente).

In [ ]:
# Definindo o schema das tabelas agregadas (meu Data Mart de vendas)
schema = [
    "data_referencia",
    "id_produto",
    "tipo_pagamento",
    "receita_total",
    "qtd_transacoes"
]

# 1) Base legada
# Essa aqui é a "fonte da verdade", ou seja, o que o pipeline deveria reproduzir
data_legacy = [
    ("2026-07-01", "PROD_A", "Pix", 1500.50, 10),
    ("2026-07-01", "PROD_B", "Credito", 3200.00, 15),  # Essa linha some na Gold
    ("2026-07-02", "PROD_A", "Pix", 1200.00, 8),
    ("2026-07-02", "PROD_C", "Debito", 850.75, 5)
]

# 2) Base Gold (a que veio do pipeline novo, no Databricks)
# Coloquei uns erros de propósito pra simular falhas reais de pipeline:
# - valor divergente de 50 centavos
# - uma linha faltando (problema de volumetria)
data_gold = [
    ("2026-07-01", "PROD_A", "Pix", 1500.00, 10),
    ("2026-07-02", "PROD_A", "Pix", 1200.00, 8),
    ("2026-07-02", "PROD_C", "Debito", 850.75, 5)
]

# Montando os DataFrames em PySpark
df_legacy = spark.createDataFrame(data_legacy, schema)
df_gold = spark.createDataFrame(data_gold, schema)

# Salvando tudo em formato Delta pra imitar o ambiente Databricks/Unity Catalog
df_legacy.write.format("delta").mode("overwrite").save("/tmp/delta/legacy_mart")
df_gold.write.format("delta").mode("overwrite").save("/tmp/delta/gold_mart")

print("Tabelas Delta criadas no storage local.")

Tabelas Delta criadas no storage local.


## 2. Motor de Reconciliação e Data Quality (DQ)

Esta etapa implementa as regras de **Analytics Engineering** para garantir a paridade dos dados antes do reapontamento do BI. O processo consiste em:
1. **Modelagem de Comparação:** Cruzar as bases utilizando as dimensões do *Data Mart* (Granularidade).
2. **Cálculo de Delta:** Medir a variação absoluta entre as métricas (Receita e Quantidade).
3. **DQ Gates (Expectations):** Classificar automaticamente cada registro. Se houver falhas de volumetria (linhas ausentes) ou divergência financeira (acima de $0.01), o registro é marcado como **FALHA**, gerando um log de auditoria que bloquearia a atualização do Power BI.

In [ ]:
# Primeiro puxo as duas tabelas Delta pra memória
df_leg = spark.read.format("delta").load("/tmp/delta/legacy_mart")
df_gld = spark.read.format("delta").load("/tmp/delta/gold_mart")

# Como as duas têm coluna com o mesmo nome, renomeio pra não virar bagunça depois do join
df_leg = df_leg.withColumnRenamed("receita_total", "receita_legado") \
               .withColumnRenamed("qtd_transacoes", "qtd_legado")

df_gld = df_gld.withColumnRenamed("receita_total", "receita_gold") \
               .withColumnRenamed("qtd_transacoes", "qtd_gold")

# A chave aqui é a granularidade da tabela: dia + produto + meio de pagamento
# Uso full_outer pra pegar tanto o que falta na Gold quanto o que "apareceu do nada"
chaves_negocio = ["data_referencia", "id_produto", "tipo_pagamento"]
df_recon = df_leg.join(df_gld, on=chaves_negocio, how="full_outer")

# Agora comparo os valores e classifico cada linha
# A tolerância de 0.01 é só pra ignorar diferença de arredondamento de centavo
df_audit = df_recon.withColumn(
    "delta_receita", round(col("receita_gold") - col("receita_legado"), 2)
).withColumn(
    "status_reconciliacao",
    when(col("receita_legado").isNull(), "FALHA: registro existe só na Gold")
    .when(col("receita_gold").isNull(), "FALHA: registro sumiu na Gold")
    .when(abs(col("delta_receita")) > 0.01, "FALHA: valor não bate")
    .otherwise("OK: bateu exato")
)

# No relatório final só me interessa o que deu errado
df_erros = df_audit.filter(col("status_reconciliacao") != "OK: bateu exato")

print("ATENÇÃO: tem divergência entre as bases, então o reapontamento do BI fica bloqueado.\n")

df_erros.select(
    "data_referencia", "id_produto",
    "receita_legado", "receita_gold",
    "delta_receita", "status_reconciliacao"
).show(truncate=False)

ATENÇÃO: tem divergência entre as bases, então o reapontamento do BI fica bloqueado.

+---------------+----------+--------------+------------+-------------+-----------------------------+
|data_referencia|id_produto|receita_legado|receita_gold|delta_receita|status_reconciliacao         |
+---------------+----------+--------------+------------+-------------+-----------------------------+
|2026-07-01     |PROD_A    |1500.5        |1500.0      |-0.5         |FALHA: valor não bate        |
|2026-07-01     |PROD_B    |3200.0        |NULL        |NULL         |FALHA: registro sumiu na Gold|
+---------------+----------+--------------+------------+-------------+-----------------------------+



## 3. Governança e Linhagem (Unity Catalog)

Em um ambiente Databricks real, a garantia de que as métricas são confiáveis não para no código. O **Unity Catalog** entra em cena para registrar a linhagem dos dados e certificar as tabelas.

* **Lineage (Linhagem):** O Unity Catalog rastrearia visualmente que a tabela `gold_mart` tem como origem o *script* `bronze_to_silver_dq.py`.
* **Descoberta:** Cientistas de dados poderiam buscar por "Receita" no catálogo e encontrar a `gold_mart` com a tag `CERTIFICADA`.
* **Data Contracts em Produção:** Se a tabela Gold não passasse nas *Expectations* do nosso motor acima, as *tags* do Unity Catalog seriam atualizadas para `QUARANTINE`, impossibilitando a leitura pelo BI.

## 4. Plano de Reapontamento do BI (Power BI / Tableau)

Assim que a engenharia de dados corrigir o *pipeline* da camada Silver e o nosso Motor de Reconciliação retornar `SUCESSO: Paridade Exata` para todas as linhas, o processo de migração do *dashboard* é executado da seguinte forma:

1. **Hot-Swap via XMLA Endpoint (Power BI Premium):** Conectar diretamente ao *dataset* publicado e alterar a *connection string* do Banco Legado para o Databricks SQL Warehouse, sem precisar baixar o `.pbix`.
2. **Refatoração no Power Query (M):** Se for um reapontamento manual, substituímos a função `Sql.Database` pela conexão `Databricks.Catalogs`, mantendo os nomes exatos das colunas.
3. **Validação Visual:** Uma última checagem nos cartões de KPI (Receita MTD, YTD) para garantir que nenhuma medida DAX quebrou devido à mudança do motor analítico por trás do modelo tabular.

## 5. Arquitetura Medalhão: Camada Bronze (Ingestão e Data Contracts)

A Camada Bronze é o ponto de aterrissagem dos dados (*Landing Zone*). Neste módulo, simulamos a extração de um fluxo contínuo de transações de um robô de operações quantitativas (via API).

**Padrões implementados:**
1. **Data Contracts:** Definição estrita do *schema* de entrada. Se a API de origem mudar o formato do dado sem avisar, o *pipeline* falha aqui, protegendo as camadas seguintes.
2. **Injeção de Caos:** O script gera dados válidos de operações de ETFs, Cripto e Pix, mas introduz intencionalmente anomalias (valores nulos e preços negativos) em 5% das linhas para testarmos nossos *DQ Gates* na Camada Silver.
3. **Imutabilidade:** Os dados são gravados no formato Delta exatamente como chegaram, mantendo o histórico bruto para reprocessamento futuro.

In [6]:
import random
import builtins  # pra recuperar o round "original" do Python
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

contract_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("asset_class", StringType(), True),
    StructField("operation_type", StringType(), True),
    StructField("volume", DoubleType(), True),
    StructField("price", DoubleType(), True)
])

def extracao_api_quant(num_records):
    data = []
    ativos = ["ETF_TOTAL_RETURN", "CRYPTO_LIQUIDITY_POOL", "FIAT_BRL"]
    operacoes = ["BUY_SYNTHETIC", "SELL_MARKET", "PIX_TRANSFER"]

    for i in range(num_records):
        registro_zoado = random.random() < 0.05

        data.append(Row(
            transaction_id=f"TXN_QT_{88000 + i}",
            timestamp=datetime.now() - timedelta(minutes=random.randint(1, 1440)),
            asset_class=random.choice(ativos) if not registro_zoado else None,
            operation_type=random.choice(operacoes),
            # builtins.round: uso o round do Python, porque o "round" solto no
            # notebook é o do pyspark.sql.functions, que devolve Column e quebra tudo
            volume=builtins.round(random.uniform(0.5, 50.0), 4),
            price=builtins.round(random.uniform(15.0, 350.0), 2) if not registro_zoado else -99.99
        ))
    return data

payload_bruto = extracao_api_quant(250)
df_bronze = spark.createDataFrame(payload_bruto, schema=contract_schema)

df_bronze.write.format("delta").mode("overwrite").save("/tmp/delta/bronze_layer")

print("Ingestão concluída. Payload gravado na camada Bronze.\n")
df_bronze.show(10, truncate=False)

Ingestão concluída. Payload gravado na camada Bronze.

+--------------+--------------------------+---------------------+--------------+-------+------+
|transaction_id|timestamp                 |asset_class          |operation_type|volume |price |
+--------------+--------------------------+---------------------+--------------+-------+------+
|TXN_QT_88000  |2026-08-03 08:37:31.543074|ETF_TOTAL_RETURN     |PIX_TRANSFER  |47.0701|214.25|
|TXN_QT_88001  |2026-08-03 16:44:31.543312|ETF_TOTAL_RETURN     |PIX_TRANSFER  |21.3413|21.53 |
|TXN_QT_88002  |2026-08-03 11:29:31.543349|CRYPTO_LIQUIDITY_POOL|SELL_MARKET   |36.647 |85.18 |
|TXN_QT_88003  |2026-08-03 19:27:31.543361|FIAT_BRL             |SELL_MARKET   |13.732 |272.03|
|TXN_QT_88004  |2026-08-03 04:50:31.543371|CRYPTO_LIQUIDITY_POOL|PIX_TRANSFER  |22.0663|237.86|
|TXN_QT_88005  |2026-08-03 03:28:31.54338 |ETF_TOTAL_RETURN     |BUY_SYNTHETIC |33.4736|264.75|
|TXN_QT_88006  |2026-08-03 06:56:31.543389|CRYPTO_LIQUIDITY_POOL|PIX_TRANSFER  |4

## 6. Arquitetura Medalhão: Camada Silver (Data Quality e Quarentena)

A Camada Silver é responsável pelo refinamento e higienização. Aqui, os dados deixam de ser um "pântano" e passam a ser confiáveis para análise.

**Estratégia de Data Quality (DQ) Implementada:**
*   **Regras de Validação:** O preço do ativo deve ser estritamente maior que zero e a classe do ativo não pode ser nula.
*   **Roteamento Dinâmico (Dead Letter Queue):** Em vez de usar um `.dropna()` que destrói o histórico de erros, utilizamos a lógica condicional do PySpark para dividir o fluxo em dois:
    1.  `silver_clean`: Tabela higienizada, tipada e pronta para agregações financeiras.
    2.  `silver_quarantine`: Tabela isolada que captura a linha com erro e anexa automaticamente o motivo da falha (`dq_failure_reason`) para auditoria da engenharia de dados.

In [7]:
from pyspark.sql.functions import col, when, current_timestamp

# Trago de volta o que foi salvo na Bronze
df_bronze = spark.read.format("delta").load("/tmp/delta/bronze_layer")

# Regra simples de qualidade: preço tem que ser positivo e a classe do ativo
# não pode vir vazia. É o mínimo pra esse dado fazer sentido depois
dq_rules = (col("price") > 0) & (col("asset_class").isNotNull())

# O que passa na regra segue pra Silver, já com carimbo de quando foi processado
df_silver_clean = df_bronze.filter(dq_rules) \
    .withColumn("silver_ingestion_at", current_timestamp())

# O que não passa não é descartado; vai pra quarentena, junto com o motivo
# da rejeição. Assim dá pra auditar depois o que a fonte mandou de errado
df_quarantine = df_bronze.filter(~dq_rules) \
    .withColumn("dq_failure_reason",
        when(col("asset_class").isNull(), "CRITICO: asset_class veio nulo")
        .when(col("price") <= 0, "CRITICO: preco zerado ou negativo")
        .otherwise("NAO_CATALOGADO: caiu na regra mas nao sei o motivo")
    ).withColumn("quarantined_at", current_timestamp())

# Gravo cada DataFrame na sua camada
df_silver_clean.write.format("delta").mode("overwrite").save("/tmp/delta/silver_layer")
df_quarantine.write.format("delta").mode("overwrite").save("/tmp/delta/quarantine_layer")

# Conferindo se a conta fecha: aprovados + quarentena tem que dar o total da Bronze
total_in = df_bronze.count()
total_clean = df_silver_clean.count()
total_quarantine = df_quarantine.count()

print("Resumo da triagem Bronze -> Silver:")
print(f"Entraram da Bronze : {total_in}")
print(f"Passaram (Silver)  : {total_clean}")
print(f"Foram p/ quarentena: {total_quarantine}\n")

# Olhando o que foi rejeitado pra ver se o motivo faz sentido
print("Amostra da quarentena:")
df_quarantine.select(
    "transaction_id", "asset_class", "price", "dq_failure_reason"
).show(truncate=False)

Resumo da triagem Bronze -> Silver:
Entraram da Bronze : 250
Passaram (Silver)  : 235
Foram p/ quarentena: 15

Amostra da quarentena:
+--------------+-----------+------+------------------------------+
|transaction_id|asset_class|price |dq_failure_reason             |
+--------------+-----------+------+------------------------------+
|TXN_QT_88137  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88145  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88158  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88168  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88173  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88188  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88198  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88218  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88229  |NULL       |-99.99|CRITICO: asset_class veio nulo|
|TXN_QT_88027  |NULL       |-99.99|CRITICO: asset_class veio n

## 7. Arquitetura Medalhão: Camada Gold (Data Mart e Camada Semântica)

A Camada Gold é o produto final de dados. Aqui, abandonamos a visão de "engenharia" (logs, pipelines, quarentenas) e adotamos a visão de **negócios**.

**Conceitos Aplicados:**
1. **Modelagem Dimensional (Kimball):** Transformamos o *timestamp* granular em uma `data_referencia` (para se conectar com uma Dimensão de Calendário no BI).
2. **Camada Semântica e Métricas Certificadas:** A regra de negócio para calcular o volume financeiro ($ \text{Volume} \times \text{Preço} $) é processada no PySpark, garantindo que o Power BI não precise rodar cálculos pesados em DAX.
3. **Data Mart por Domínio:** O resultado é a tabela `Fato_Operacoes_Quantitativas`, agregada por Data, Ativo e Tipo de Operação, pronta para consumo gerencial.

In [8]:
from pyspark.sql.functions import col, sum, count, to_date, round

# Puxo de volta o que já passou pela triagem na Silver
df_silver = spark.read.format("delta").load("/tmp/delta/silver_layer")

# Aqui preparo as colunas que vou agregar:
# - extraio só a data do timestamp (o BI não precisa de hora/minuto)
# - calculo o volume financeiro (quantidade x preço), que é o que interessa no fim do dia
df_mart = df_silver.withColumn("data_referencia", to_date(col("timestamp"))) \
                   .withColumn("volume_financeiro_brl", col("volume") * col("price"))

# Agregação da tabela fato: agrupo por dia, classe do ativo e tipo de operação
# e somo as métricas principais
df_gold = df_mart.groupBy("data_referencia", "asset_class", "operation_type") \
    .agg(
        count("transaction_id").alias("qtd_transacoes"),
        round(sum("volume"), 4).alias("volume_ativos_negociados"),
        round(sum("volume_financeiro_brl"), 2).alias("total_financeiro_movimentado")
    ) \
    .orderBy("data_referencia", "asset_class", "operation_type")

# Salvo na Gold — a partir daqui é essa tabela que o Power BI vai consumir
df_gold.write.format("delta").mode("overwrite").save("/tmp/delta/gold_layer")

print("Tabela fato gravada na camada Gold.\n")
print("Esse é o formato final que vai pro BI:\n")
df_gold.show(truncate=False)

Tabela fato gravada na camada Gold.

Esse é o formato final que vai pro BI:

+---------------+---------------------+--------------+--------------+------------------------+----------------------------+
|data_referencia|asset_class          |operation_type|qtd_transacoes|volume_ativos_negociados|total_financeiro_movimentado|
+---------------+---------------------+--------------+--------------+------------------------+----------------------------+
|2026-08-02     |CRYPTO_LIQUIDITY_POOL|BUY_SYNTHETIC |2             |78.0113                 |8642.01                     |
|2026-08-02     |CRYPTO_LIQUIDITY_POOL|SELL_MARKET   |3             |38.997                  |6450.82                     |
|2026-08-02     |FIAT_BRL             |PIX_TRANSFER  |1             |11.6755                 |217.16                      |
|2026-08-03     |CRYPTO_LIQUIDITY_POOL|BUY_SYNTHETIC |23            |697.5306                |104492.09                   |
|2026-08-03     |CRYPTO_LIQUIDITY_POOL|PIX_TRANSFER  |2